
# Rainfall Analysis and Planting Advisory

## Week 2 Data Analytics Assignment

**Project:** Analysis of monthly rainfall patterns across five Kenyan counties.

**Objective:** Clean the rainfall data, analyse rainfall patterns using NumPy, identify the wettest months and dry months, and recommend two-month planting windows.

**Tools:** Python, NumPy, CSV and Google Colab.


In [61]:
import numpy as np
import csv

In [62]:

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]


In [63]:

def clean_machakos(raw_values, month_names):

    # Step 1: Collect all valid rainfall values
    valid_values = []

    for value in raw_values:
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            continue

    # Step 2: Calculate the average of valid values
    average_rainfall = round(sum(valid_values) / len(valid_values))

    # Step 3: Replace invalid values with the average
    cleaned_values = []
    imputed_months = []

    for month, value in zip(month_names, raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(average_rainfall)
            imputed_months.append(month)

    return cleaned_values, imputed_months, average_rainfall


In [64]:

machakos_cleaned, imputed_months, average_rainfall = clean_machakos(
    machakos_raw, months
)

print("Valid rainfall average:", average_rainfall, "mm")
print("Imputed months:", imputed_months)
print("Cleaned Machakos rainfall:", machakos_cleaned)


Valid rainfall average: 68 mm
Imputed months: ['April', 'September']
Cleaned Machakos rainfall: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [65]:

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95], # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]  # Mombasa
]


In [66]:
# Combine the four counties with cleaned Machakos data
R = np.array(clean_rows + [machakos_cleaned])

print("Combined Rainfall Array:")
print(R)

print("\nArray shape:", R.shape)
print("Array data type:", R.dtype)

Combined Rainfall Array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array data type: int64


In [67]:

# Calculate total annual rainfall for each county
annual_totals = R.sum(axis=1)

# Calculate average rainfall for each month across all counties
monthly_averages = R.mean(axis=0).round(1)

print("Annual Rainfall Totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage Monthly Rainfall Across All Counties:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average} mm")


Annual Rainfall Totals:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average Monthly Rainfall Across All Counties:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


In [68]:

# Find the index of the wettest month for each county
wettest_indices = np.argmax(R, axis=1)

# Get the rainfall amount for each county's wettest month
wettest_rainfall = R[np.arange(len(counties)), wettest_indices]

# Convert month indices into month names
wettest_months = np.array(months)[wettest_indices]

# Display results using zip()
print("Wettest Month for Each County:")

for county, month, rainfall in zip(
    counties, wettest_months, wettest_rainfall
):
    print(f"{county}: {month} ({rainfall} mm)")


Wettest Month for Each County:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


In [69]:

# Create a boolean mask for rainfall below 50 mm
dry_mask = R < 50

# Count dry months for each county
dry_month_counts = dry_mask.sum(axis=1)

print("Number of Dry Months per County:")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")


Number of Dry Months per County:
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


In [70]:

# Calculate combined rainfall for consecutive two-month periods
two_month_totals = R[:, :-1] + R[:, 1:]

# Find the position of the highest two-month rainfall for each county
planting_indices = np.argmax(two_month_totals, axis=1)

# Retrieve the highest combined rainfall for each county
planting_rainfall = two_month_totals[
    np.arange(len(counties)), planting_indices
]

# Identify the starting and ending months
starting_months = np.array(months)[planting_indices]
ending_months = np.array(months)[planting_indices + 1]

# Create the planting window names
planting_windows = [
    f"{start}–{end}"
    for start, end in zip(starting_months, ending_months)
]

# Display results
print("Recommended Two-Month Planting Windows:")

for county, window, rainfall in zip(
    counties, planting_windows, planting_rainfall
):
    print(f"{county}: {window} ({rainfall} mm)")


Recommended Two-Month Planting Windows:
Nairobi: April–May (395 mm)
Nakuru: April–May (265 mm)
Kisumu: April–May (420 mm)
Mombasa: April–May (390 mm)
Machakos: November–December (278 mm)


In [71]:

# Define the required CSV columns
fieldnames = [
    "county",
    "annual_total_mm",
    "wettest_month",
    "dry_months",
    "planting_window"
]

# Prepare one record for each county
summary_data = []

for county, total, wettest, dry_count, window in zip(
    counties,
    annual_totals,
    wettest_months,
    dry_month_counts,
    planting_windows
):
    summary_data.append({
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": wettest,
        "dry_months": int(dry_count),
        "planting_window": window
    })

# Write the records into the CSV file
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_data)

print("rainfall_summary.csv created successfully!")
print("Number of county records:", len(summary_data))


rainfall_summary.csv created successfully!
Number of county records: 5


In [72]:

with open("rainfall_summary.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    print("CSV Columns:", reader.fieldnames)

    csv_records = list(reader)

    print("Number of records:", len(csv_records))

    for record in csv_records:
        print(record)


CSV Columns: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
Number of records: 5
{'county': 'Nairobi', 'annual_total_mm': '1045', 'wettest_month': 'April', 'dry_months': '4', 'planting_window': 'April–May'}
{'county': 'Nakuru', 'annual_total_mm': '916', 'wettest_month': 'April', 'dry_months': '3', 'planting_window': 'April–May'}
{'county': 'Kisumu', 'annual_total_mm': '1371', 'wettest_month': 'April', 'dry_months': '0', 'planting_window': 'April–May'}
{'county': 'Mombasa', 'annual_total_mm': '1075', 'wettest_month': 'May', 'dry_months': '2', 'planting_window': 'April–May'}
{'county': 'Machakos', 'annual_total_mm': '811', 'wettest_month': 'November', 'dry_months': '6', 'planting_window': 'November–December'}



## Planting Advisory

The analysis identifies April–May as the recommended two-month rainfall window for Nairobi, Nakuru, Kisumu and Mombasa, while Machakos has its strongest rainfall window in November–December. Kisumu recorded the highest annual rainfall at 1,371 mm, whereas Machakos recorded the lowest at 811 mm. Machakos also experienced the highest number of dry months, with six months receiving less than 50 mm of rainfall. The NGO should use these county-specific rainfall patterns to guide preliminary planting SMS messages and encourage farmers to prepare for the identified rainfall windows. However, farmers should also consider local weather forecasts, soil conditions and available water before planting.



## Data Quality Notes

The original Machakos rainfall data contained two invalid entries: "N/A" for April and a blank value for September. These entries were replaced with 68 mm, calculated by rounding the average of the ten valid Machakos rainfall readings. The other four counties' readings were used as supplied without modification. The analysis therefore contains two imputed values, which may affect Machakos rainfall totals and its recommended planting window. Additionally, the dataset covers only one year, so the identified rainfall patterns should not be treated as a guarantee of future seasonal conditions.


In [73]:

%%writefile README.md

# Rainfall Analysis and Planting Advisory

## Project Summary

This project analyses monthly rainfall data from five Kenyan counties using Python and NumPy. It cleans invalid Machakos rainfall readings, calculates rainfall statistics, identifies dry months and recommends two-month planting windows.

## How to Run

1. Open `rainfall_analysis.ipynb` in Google Colab or Jupyter Notebook.
2. Ensure NumPy is installed using `pip install -r requirements.txt`.
3. Run all notebook cells from top to bottom to reproduce the analysis and generate `rainfall_summary.csv`.

## Planting Advisory

- Nairobi, Nakuru, Kisumu and Mombasa have April–May as their recommended two-month rainfall window.
- Machakos has November–December as its recommended two-month rainfall window.
- Farmers should consider local weather forecasts, soil conditions and water availability before planting.

## Project Files

- `rainfall_analysis.ipynb` – Data cleaning and rainfall analysis.
- `rainfall_summary.csv` – Summary of rainfall results for all five counties.
- `requirements.txt` – Python dependencies.


Overwriting README.md


In [74]:

%%writefile requirements.txt
numpy


Overwriting requirements.txt


In [75]:
# Check README.md
with open("README.md", "r") as file:
    print("README CONTENT:")
    print(file.read())

# Check requirements.txt
with open("requirements.txt", "r") as file:
    print("\nREQUIREMENTS CONTENT:")
    print(file.read())

README CONTENT:

# Rainfall Analysis and Planting Advisory

## Project Summary

This project analyses monthly rainfall data from five Kenyan counties using Python and NumPy. It cleans invalid Machakos rainfall readings, calculates rainfall statistics, identifies dry months and recommends two-month planting windows.

## How to Run

1. Open `rainfall_analysis.ipynb` in Google Colab or Jupyter Notebook.
2. Ensure NumPy is installed using `pip install -r requirements.txt`.
3. Run all notebook cells from top to bottom to reproduce the analysis and generate `rainfall_summary.csv`.

## Planting Advisory

- Nairobi, Nakuru, Kisumu and Mombasa have April–May as their recommended two-month rainfall window.
- Machakos has November–December as its recommended two-month rainfall window.
- Farmers should consider local weather forecasts, soil conditions and water availability before planting.

## Project Files

- `rainfall_analysis.ipynb` – Data cleaning and rainfall analysis.
- `rainfall_summary.cs